# 📔 Notebook de Estudio: Inteligencia Artificial Avanzada

**Autor:** Apuntes de Clase (Consolidados)  
**Temas:** Búsqueda · ML Clásico · Audio · Redes Neuronales (CNN/MLP) · YOLO · ResNet

---
## 🟢 Módulo 1: Búsqueda y Optimización
Fundamentos de cómo las máquinas exploran soluciones.

### 1.1 Algoritmo A\* (Búsqueda Informada)

El algoritmo A\* utiliza una función de coste para encontrar la salida de un laberinto:

$$f(n) = g(n) + h(n)$$

| Componente | Descripción |
|:----------:|-------------|
| $g(n)$ | Coste **real** desde el inicio al nodo actual |
| $h(n)$ | **Heurística** – estimación al objetivo (ej: Distancia Manhattan) |

**Ejemplo de implementación (Lógica `pyamaze`):**

In [ ]:
def h(celda1, celda2):
    x1, y1 = celda1
    x2, y2 = celda2
    return abs(x1 - x2) + abs(y1 - y2)  # Distancia Manhattan

In [ ]:
import heapq

def a_star(maze, start, end):
    """
    A* search on a grid maze.
    maze: 2D list where 0=free, 1=wall
    Returns the path as a list of (row, col) tuples, or None if no path.
    """
    def h(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])  # Distancia Manhattan

    open_set = []
    heapq.heappush(open_set, (0, start))

    came_from = {}
    g = {start: 0}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == end:
            # Reconstruir camino
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            return [start] + path[::-1]

        r, c = current
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nb = (r + dr, c + dc)
            nr, nc = nb
            if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]) and maze[nr][nc] == 0:
                new_g = g[current] + 1
                if nb not in g or new_g < g[nb]:
                    g[nb] = new_g
                    f = new_g + h(nb, end)
                    heapq.heappush(open_set, (f, nb))
                    came_from[nb] = current

    return None  # No hay camino

# --- Ejemplo ---
maze = [
    [0, 0, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0],
]
path = a_star(maze, (0,0), (4,4))
print("Camino encontrado:", path)

### 1.2 Programación Genética (N-Reinas)

Busca soluciones mediante evolución biológica simulada.

| Concepto | Descripción |
|----------|-------------|
| **Individuo** | Un tablero representado por un vector |
| **Fitness** | Cantidad de ataques entre reinas (objetivo: `0`) |
| **Operaciones** | Selección de los más aptos → Cruce (*Crossover*) → Mutación aleatoria |

In [ ]:
import random

def calcular_fitness(individuo):
    """Cuenta cuántos pares de reinas se atacan. Objetivo: 0."""
    ataques = 0
    n = len(individuo)
    for i in range(n):
        for j in range(i + 1, n):
            if individuo[i] == individuo[j] or abs(individuo[i] - individuo[j]) == abs(i - j):
                ataques += 1
    return ataques

def genetic_8_queens(n=8, poblacion_size=100):
    # 1. Población inicial aleatoria
    poblacion = [[random.randint(0, n-1) for _ in range(n)] for _ in range(poblacion_size)]

    for generacion in range(1000):
        # Ordenar por fitness (menor = mejor)
        poblacion = sorted(poblacion, key=lambda ind: calcular_fitness(ind))

        if calcular_fitness(poblacion[0]) == 0:
            print(f"✅ Solución encontrada en generación {generacion}: {poblacion[0]}")
            return poblacion[0]

        # 2. Selección + Crossover
        nueva_poblacion = poblacion[:20]  # élite
        while len(nueva_poblacion) < poblacion_size:
            padre1, padre2 = random.sample(poblacion[:50], 2)
            punto_cruce    = random.randint(1, n - 1)
            hijo           = padre1[:punto_cruce] + padre2[punto_cruce:]

            # 3. Mutación (5%)
            if random.random() < 0.05:
                hijo[random.randint(0, n-1)] = random.randint(0, n-1)
            nueva_poblacion.append(hijo)

        poblacion = nueva_poblacion

    print(f"❌ Sin solución tras 1000 generaciones. Mejor fitness: {calcular_fitness(poblacion[0])}")
    return None

solucion = genetic_8_queens(8)
if solucion:
    print(f"Ataques: {calcular_fitness(solucion)}")

---
## 🔵 Módulo 2: Machine Learning Clásico
Modelos basados en estadística y geometría para datos estructurados.

### 2.1 Clasificación Tabular (Apple & Iris)

Aprendizaje para predecir etiquetas basadas en características (*features*).

- **KNN (K-Nearest Neighbors):** Clasifica un punto mirando a sus *K* vecinos más cercanos.
- **Decision Tree:** Árbol de decisiones tipo «Sí/No» basado en condiciones de los datos.
- **Preprocesamiento:** Es vital el uso de `StandardScaler` para que todas las variables tengan la misma escala numérica.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class KNN:
    """K-Nearest Neighbors implementado desde cero."""

    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def _euclidean_distance(self, a, b):
        return np.sqrt(np.sum((a - b) ** 2))

    def predict_one(self, x):
        distances = np.array([self._euclidean_distance(x, xt) for xt in self.X_train])
        k_idx     = distances.argsort()[:self.k]
        k_labels  = self.y_train[k_idx]
        values, counts = np.unique(k_labels, return_counts=True)
        return values[counts.argmax()]  # clase mayoritaria

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

    def accuracy(self, y_true, y_pred):
        return np.mean(y_true == y_pred) * 100

# --- Ejemplo con datos sintéticos (sin necesidad del CSV) ---
np.random.seed(42)
X_demo = np.vstack([
    np.random.randn(50, 2) + [2, 2],   # clase "good"
    np.random.randn(50, 2) + [-2, -2], # clase "bad"
])
y_demo = np.array(['good'] * 50 + ['bad'] * 50)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_demo)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_demo, test_size=0.2, random_state=42)

model = KNN(k=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"KNN Accuracy: {model.accuracy(y_test, y_pred):.1f}%")

### 2.2 Clasificación de Audio (Vocales)

Conversión de sonido en vectores matemáticos.

- **Extracción:** Se usa la **FFT** para obtener las frecuencias dominantes (Formantes F1 y F2).
- **Modelado:** Se utiliza un **SVM** (*Support Vector Machine*) con Kernel **RBF** para crear fronteras de decisión complejas entre las vocales.

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import classification_report

# --- Pipeline completo: FFT → vectores → SVM ---
# (Requiere wav2vec.py y los archivos .wav/.json del proyecto Vowels)
#
# from wav2vec import cutvowel, wav2vec
# import json
#
# with open("Edu.json") as f:
#     data = json.load(f)
#
# labels, vectors = [], []
# for entry in data:
#     Fs, cut = cutvowel("beppo.wav", entry["start"], entry["end"])
#     vec = wav2vec(cut, Fs)            # FFT → [F1, F2, F3]
#     labels.append(entry["vocal"])
#     vectors.append(vec)
#
# X = np.array(vectors)
# y = np.array(labels)

# --- Demo con datos sintéticos representando F1/F2 de 5 vocales ---
np.random.seed(0)
formants = {'A': [800,1200], 'E': [500,1800], 'I': [300,2300], 'O': [500,900], 'U': [300,700]}
X_list, y_list = [], []
for vocal, (f1, f2) in formants.items():
    for _ in range(20):
        X_list.append([f1 + np.random.randn()*40, f2 + np.random.randn()*60])
        y_list.append(vocal)

X = np.array(X_list)
y = np.array(y_list)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SVM con kernel RBF — crea fronteras de decisión no lineales
svm = SVC(kernel='rbf', C=10, gamma='scale', decision_function_shape='ovr')

# Validación Leave-One-Out
loo    = LeaveOneOut()
scores = cross_val_score(svm, X_scaled, y, cv=loo, scoring='accuracy')
print(f"LOO Accuracy: {scores.mean()*100:.1f}%  ({int(scores.sum())}/{len(scores)} correctas)")

svm.fit(X_scaled, y)
print(classification_report(y, svm.predict(X_scaled), target_names=sorted(set(y))))

---
## 🔴 Módulo 3: Redes Neuronales (Deep Learning)
Construcción de capas inspiradas en el cerebro para patrones complejos.

### 3.1 Diccionario Lógico de Capas

| Capa | Función |
|------|---------|
| `Conv2d` | Escáner que busca rasgos visuales (bordes, texturas) |
| `ReLU` | Filtro que elimina señales negativas (activa solo lo importante) |
| `MaxPool2d` | Resume la imagen, quedándose con el rasgo más fuerte de cada zona |
| `Flatten` | Estira la matriz 2D en una fila para poder realizar la votación final |
| `Linear` | Capas densas donde cada neurona «vota» por una clase final |

In [ ]:
import torch
import torch.nn as nn

class MNISTModel(nn.Module):
    """
    CNN para MNIST (1×28×28 → 10 clases).
    Muestra en acción cada capa del diccionario.
    """
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
            # ── Bloque convolucional 1 ──────────────────────────────
            nn.Conv2d(1, 32, kernel_size=3),   # Escáner: busca rasgos (bordes, texturas)
            nn.ReLU(),                          # Elimina activaciones negativas
            nn.MaxPool2d(2, 2),                 # Resume: queda con el rasgo más fuerte

            # ── Bloque convolucional 2 ──────────────────────────────
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # ── Clasificador ────────────────────────────────────────
            nn.Flatten(),                       # 2D → vector 1D
            nn.Linear(1600, 128),               # Cada neurona "vota"
            nn.ReLU(),
            nn.Linear(128, 10),                 # 10 clases finales
        )

    def forward(self, x):
        return self.seq(x)

model = MNISTModel()
print(model)

# Traza de formas por capa
x = torch.zeros(1, 1, 28, 28)
print(f"\nInput:          {x.shape}")
for layer in model.seq:
    x = layer(x)
    print(f"{type(layer).__name__:<15s} → {x.shape}")

### 3.2 El Ciclo de Entrenamiento

```
Input ──► FeedForward ──► Predicción
                               │
                          Loss Function  ← mide el error
                               │
                        Backpropagation  ← propaga el gradiente
                               │
                      Actualización de W  ← ajusta los pesos
```

1. **FeedForward:** Los datos viajan del *input* al *output* para dar una predicción.
2. **Loss Function:** Mide la distancia entre la predicción y la realidad (castigo).
3. **Backpropagation:** El error viaja hacia atrás para ajustar los pesos $W$ mediante el gradiente.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def train_loop(dataloader, model, loss_fn, optimizer):
    """FeedForward → Loss → Backpropagation → actualiza W."""
    model.train()
    total_loss = 0.0
    for X, y in dataloader:
        pred = model(X)             # 1. FeedForward
        loss = loss_fn(pred, y)     # 2. Loss Function (castigo)

        optimizer.zero_grad()
        loss.backward()             # 3. Backpropagation (gradiente)
        optimizer.step()            # 4. Actualiza pesos W

        total_loss += loss.item()
    return total_loss / len(dataloader)

def test_loop(dataloader, model, loss_fn):
    model.eval()
    correct, total_loss = 0, 0.0
    with torch.no_grad():
        for X, y in dataloader:
            pred        = model(X)
            total_loss += loss_fn(pred, y).item()
            correct    += (pred.argmax(1) == y).sum().item()
    acc = 100.0 * correct / len(dataloader.dataset)
    return total_loss / len(dataloader), acc

# --- Demo rápida ---
torch.manual_seed(42)
X_demo = torch.randn(200, 1600)
y_demo = torch.randint(0, 10, (200,))
loader = DataLoader(TensorDataset(X_demo, y_demo), batch_size=32, shuffle=True)

simple_model = nn.Sequential(nn.Linear(1600, 128), nn.ReLU(), nn.Linear(128, 10))
loss_fn      = nn.CrossEntropyLoss()
optimizer    = torch.optim.SGD(simple_model.parameters(), lr=0.01)

for epoch in range(1, 4):
    tr_loss        = train_loop(loader, simple_model, loss_fn, optimizer)
    te_loss, acc   = test_loop(loader,  simple_model, loss_fn)
    print(f"Epoch {epoch}  Loss: {tr_loss:.4f}  Acc: {acc:.1f}%")

---
## 🟡 Módulo 4: Arquitecturas Avanzadas

### 4.1 YOLO (Detección en Tiempo Real)

Modelo que predice cajas delimitadoras e IDs en una sola pasada.

- **Tracking (ByteTrack):** Mantiene la identidad de un objeto entre fotogramas.
- **PolygonZone:** Define un área específica para contar o detectar intrusos.

In [ ]:
import cv2
import argparse
import numpy as np
import supervision as sv
from ultralytics import YOLO
import torch

# ── Zona de detección (polígono sobre el vídeo) ────────────────────────────
POLYGON = np.array([
    [287, 509], [289, 574], [2,   585], [3,   852],
    [179, 899], [285, 943], [1679,941], [1681,580], [551, 499]
])
CLASSES = [2, 3]   # COCO: 2=car, 3=motorcycle

# ── Modelo + Tracker ───────────────────────────────────────────────────────
model   = YOLO("yolo11n.pt")
tracker = sv.ByteTrack(minimum_consecutive_frames=3)   # ByteTrack: mantiene IDs entre frames
tracker.reset()

# ── Zona y anotadores ─────────────────────────────────────────────────────
polygon_zone    = sv.PolygonZone(polygon=POLYGON, triggering_anchors=(sv.Position.CENTER,))
box_annotator   = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_position=sv.Position.TOP_LEFT)
trace_annotator = sv.TraceAnnotator(trace_length=60)   # traza de trayectoria


def main(video_file_path):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device: {device}")

    frame_generator = sv.get_video_frames_generator(source_path=video_file_path)

    for i, frame in enumerate(frame_generator):
        print(f"Frame {i}")

        # 1. Inferencia YOLO (una sola pasada)
        results    = model(frame, device=device, verbose=False, imgsz=1280)[0]
        detections = sv.Detections.from_ultralytics(results)

        # 2. Filtros: zona + clases de interés
        detections = detections[polygon_zone.trigger(detections=detections)]
        detections = detections[np.isin(detections.class_id, CLASSES)]

        # 3. Tracking: asigna/mantiene IDs persistentes
        detections = tracker.update_with_detections(detections)

        labels = [f"#{tid}" for tid in detections.tracker_id]

        # 4. Anotación del frame
        annotated = frame.copy()
        annotated = sv.draw_polygon(scene=annotated, polygon=POLYGON,
                                    color=sv.Color.RED, thickness=2)
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
        annotated = trace_annotator.annotate(scene=annotated, detections=detections)

        cv2.imshow("Processed Video", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()


# ── Uso ───────────────────────────────────────────────────────────────────
# main("ruta/al/video.mp4")
print("Ejecuta: main('ruta/al/video.mp4')  o desde terminal: python app.py --video_file_path video.mp4")

### 4.2 ResNet-18 (Conexiones Residuales)

Diseño para redes muy profundas que evita que el aprendizaje se detenga.

```
x ──► Conv → BN → ReLU → Conv → BN ──► (+) ──► ReLU
 ╲                                    ↗
  ╲────── Skip Connection ───────────╯
```

- **Skip Connection:** Permite que la información «salte» capas para que el gradiente llegue al principio de la red.
- **Optimización Adam:** Algoritmo inteligente que ajusta la velocidad de aprendizaje (*Learning Rate*) dinámicamente.

In [ ]:
import torch
import torch.nn as nn


# ── Custom CrossEntropyLoss ────────────────────────────────────────────────
class CrossEntropyLoss(nn.Module):
    """Log-sum-exp estable: sin nn.CrossEntropyLoss."""
    def forward(self, logits, targets):
        shift     = logits.max(dim=1, keepdim=True).values
        log_probs = logits - shift - (logits - shift).exp().sum(dim=1, keepdim=True).log()
        return -log_probs[torch.arange(len(targets)), targets].mean()


# ── Custom Adam ────────────────────────────────────────────────────────────
class Adam(torch.optim.Optimizer):
    """Kingma & Ba (2015) desde cero: momentos m/v + bias correction."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None: continue
                g = p.grad.clone()
                if wd: g.add_(p, alpha=wd)
                s = self.state[p]
                if not s:
                    s['t'] = 0
                    s['m'] = torch.zeros_like(p)
                    s['v'] = torch.zeros_like(p)
                s['t'] += 1
                t = s['t']
                s['m'].mul_(b1).add_(g, alpha=1 - b1)
                s['v'].mul_(b2).addcmul_(g, g, value=1 - b2)
                m_hat = s['m'] / (1 - b1 ** t)
                v_hat = s['v'] / (1 - b2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)


# ── BasicBlock (Skip Connection) ───────────────────────────────────────────
class BasicBlock(nn.Module):
    """Bloque residual: x → Conv→BN→ReLU→Conv→BN → (+skip) → ReLU"""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1,      padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        # Projection shortcut cuando cambian las dimensiones
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))   # ← Skip Connection


# ── ResNet-18 (CIFAR-10 adaptado) ─────────────────────────────────────────
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),  # 3×3 (no 7×7)
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),             # sin MaxPool
        )
        self.layer1 = self._make_layer(64,  64,  2, stride=1)
        self.layer2 = self._make_layer(64,  128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d((1, 1))
        self.fc     = nn.Linear(512, num_classes)

    @staticmethod
    def _make_layer(in_ch, out_ch, n, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, n):
            layers.append(BasicBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.fc(torch.flatten(self.pool(x), 1))


# ── Verificación rápida ────────────────────────────────────────────────────
model     = ResNet18(num_classes=10)
loss_fn   = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

x      = torch.randn(4, 3, 32, 32)
y      = torch.randint(0, 10, (4,))
logits = model(x)
loss   = loss_fn(logits, y)
loss.backward()
optimizer.step()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Output shape : {logits.shape}")
print(f"Loss         : {loss.item():.4f}")
print(f"Parámetros   : {total_params:,}")

---
## 📂 Organización de Archivos del Proyecto

Para que los ejercicios funcionen, mantén esta estructura:

| Archivo | Rol |
|---------|-----|
| `main.py` / `heart_disease.ipynb` | Scripts de ejecución principal |
| `classes.py` | Definición de modelos (ResNet, MLP, Custom Adam) |
| `wav2vec.py` | Procesamiento de señales de audio |
| `pyamaze.py` | Motor gráfico para laberintos |

> **Sugerencia de estudio:** Revisa las **Matrices de Confusión** de cada ejercicio; son el mapa que te dirá exactamente dónde se está confundiendo tu modelo (ej: si confunde la vocal *O* con la *U*, o el número *7* con el *1*).